In [6]:
# CVE2Patch - CVE Selection
# Cella 1: Importazione delle librerie

import requests
import pandas as pd
import json
import re
import tomllib
import yaml
import hashlib

from pathlib import Path
from datetime import datetime, timezone

print("Librerie importate correttamente.")

Librerie importate correttamente.


In [7]:
# Cella 2: Configurazione del progetto

BASE_DIR = Path.cwd() / "CVE2Patch"

RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
OUTPUT_DIR = BASE_DIR / "output"

# Creazione delle directory
for directory in [RAW_DIR, PROCESSED_DIR, OUTPUT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# Parametri di selezione
START_DATE = "2019-01-01"
END_DATE = datetime.now(timezone.utc).strftime("%Y-%m-%d")

PLATFORM = "linux"
ATTACK_VECTORS = ["NETWORK", "LOCAL"]
EXCLUDE_KERNEL = True

print("Configurazione completata.")
print(f"Periodo di raccolta: {START_DATE} - {END_DATE}")
print(f"Directory del progetto: {BASE_DIR}")

Configurazione completata.
Periodo di raccolta: 2019-01-01 - 2026-11-17
Directory del progetto: d:\leopo\Desktop\Leonardo\Uni\Tesi\Work\CVE\CVE2Patch


In [8]:
import gzip

NVD_DIR = RAW_DIR / "nvd_feeds"
NVD_DIR.mkdir(parents=True, exist_ok=True)

NVD_BASE_URL = "https://nvd.nist.gov/feeds/json/cve/2.0"

NVD_FILES = []

for year in range(2019, int(END_DATE[:4]) + 1):

    filename = f"nvdcve-2.0-{year}.json.gz"
    file_path = NVD_DIR / filename

    if not file_path.exists():

        print(f"Download dataset {year}...")

        url = f"{NVD_BASE_URL}/{filename}"
        temp_path = file_path.with_suffix(".part")

        with requests.get(url, stream=True, timeout=120) as response:
            response.raise_for_status()

            with temp_path.open("wb") as f:
                for chunk in response.iter_content(1024 * 1024):
                    f.write(chunk)

        temp_path.replace(file_path)

    else:
        print(f"Dataset {year} già disponibile.")

    NVD_FILES.append(file_path)

print("\nAcquisizione completata.")
print(f"Dataset annuali disponibili: {len(NVD_FILES)}")

Download dataset 2019...
Download dataset 2020...
Download dataset 2021...
Download dataset 2022...
Download dataset 2023...
Download dataset 2024...
Download dataset 2025...
Download dataset 2026...

Acquisizione completata.
Dataset annuali disponibili: 8


In [9]:
# Cella 4: Estrazione degli attributi e filtraggio

def extract_cvss(cve):
    metrics = cve.get("metrics", {})

    for version in ["cvssMetricV40", "cvssMetricV31",
                    "cvssMetricV30", "cvssMetricV2"]:

        entries = metrics.get(version, [])

        if entries:
            metric = next(
                (m for m in entries if m.get("type") == "Primary"),
                entries[0]
            )

            data = metric.get("cvssData", {})

            return (
                data.get("baseScore"),
                data.get("attackVector", data.get("accessVector")),
                data.get("version")
            )

    return None, None, None


def extract_cpe(obj):
    results = []

    if isinstance(obj, dict):
        for match in obj.get("cpeMatch", []):
            if match.get("criteria"):
                results.append(match["criteria"])

        for value in obj.values():
            if isinstance(value, (dict, list)):
                results.extend(extract_cpe(value))

    elif isinstance(obj, list):
        for item in obj:
            results.extend(extract_cpe(item))

    return results


records = []

for file_path in NVD_FILES:

    with gzip.open(file_path, "rt", encoding="utf-8") as f:
        data = json.load(f)

    for item in data["vulnerabilities"]:

        cve = item["cve"]

        description = next(
            (d["value"] for d in cve.get("descriptions", [])
             if d["lang"] == "en"),
            ""
        )

        cvss, attack_vector, cvss_version = extract_cvss(cve)

        cpes = sorted(set(
            extract_cpe(cve.get("configurations", []))
        ))

        cwes = sorted({
            d["value"]
            for w in cve.get("weaknesses", [])
            for d in w.get("description", [])
        })

        references = [
            r["url"] for r in cve.get("references", [])
        ]

        # Ricerca preliminare di evidenze Linux
        text = (description + " " + " ".join(cpes)).lower()

        linux_match = bool(re.search(
            r"linux|ubuntu|debian|fedora|centos|redhat|"
            r"red hat|suse|rocky_linux|almalinux",
            text
        ))

        # Identificazione preliminare delle CVE kernel
        kernel_match = (
            any(":linux:linux_kernel:" in cpe for cpe in cpes)
            or "in the linux kernel" in description.lower()
            or cve.get("sourceIdentifier") == "cve@kernel.org"
        )

        records.append({
            "cve_id": cve["id"],
            "published": cve.get("published"),
            "description": description,
            "status": cve.get("vulnStatus"),
            "cvss": cvss,
            "cvss_version": cvss_version,
            "attack_vector": attack_vector,
            "cwe": json.dumps(cwes),
            "cpe": json.dumps(cpes),
            "references": json.dumps(references),
            "linux_match": linux_match or kernel_match,
            "kernel": kernel_match
        })


# Creazione del DataFrame
df_all = pd.DataFrame(records)
df_all = df_all.drop_duplicates(subset="cve_id")

# Applicazione dei filtri
df_valid = df_all[df_all["status"] != "Rejected"]

df_av = df_valid[
    df_valid["attack_vector"].isin(ATTACK_VECTORS)
]

df_linux = df_av[df_av["linux_match"]]

# Separazione delle vulnerabilità kernel
df_kernel = df_linux[df_linux["kernel"]].copy()

df_candidates = (
    df_linux[~df_linux["kernel"]].copy()
    if EXCLUDE_KERNEL else df_linux.copy()
)

# CVE per cui non abbiamo ancora evidenze Linux
df_unknown = df_av[~df_av["linux_match"]].copy()

# Salvataggio locale
df_all.to_csv(PROCESSED_DIR / "all_cves.csv", index=False)
df_kernel.to_csv(PROCESSED_DIR / "kernel_cves.csv", index=False)
df_unknown.to_csv(PROCESSED_DIR / "platform_unknown.csv", index=False)
df_candidates.to_csv(PROCESSED_DIR / "candidates.csv", index=False)

# Risultati
print(f"CVE totali:                  {len(df_all)}")
print(f"CVE non rigettate:           {len(df_valid)}")
print(f"CVE Network / Local:         {len(df_av)}")
print(f"CVE con evidenze Linux:      {len(df_linux)}")
print(f"CVE kernel:                 {len(df_kernel)}")
print(f"CVE senza evidenze Linux:    {len(df_unknown)}")
print(f"CVE candidate:               {len(df_candidates)}")

display(df_candidates.head())

CVE totali:                  298245
CVE non rigettate:           287993
CVE Network / Local:         274102
CVE con evidenze Linux:      29429
CVE kernel:                 17951
CVE senza evidenze Linux:    244673
CVE candidate:               11478


,cve_id,published,description,status,cvss,cvss_version,attack_vector,cwe,cpe,references,linux_match,kernel
1,CVE-2019-3500,2019-01-02T07:29:00.197,"aria2c in aria2 1.33.1, when --log is used, ca...",Modified,7.8,3.1,LOCAL,"[""CWE-532""]","[""cpe:2.3:a:aria2_project:aria2:1.33.1:*:*:*:*...","[""https://github.com/aria2/aria2/issues/1329"",...",True,False
46,CVE-2019-0553,2019-01-08T21:29:00.923,An information disclosure vulnerability exists...,Modified,5.5,3.0,LOCAL,"[""NVD-CWE-noinfo""]","[""cpe:2.3:o:microsoft:windows_10:1703:*:*:*:*:...","[""http://www.securityfocus.com/bid/106412"", ""h...",True,False
81,CVE-2019-5716,2019-01-08T23:29:00.280,"In Wireshark 2.6.0 to 2.6.5, the 6LoWPAN disse...",Modified,5.5,3.0,LOCAL,"[""CWE-20""]","[""cpe:2.3:a:wireshark:wireshark:*:*:*:*:*:*:*:...","[""http://lists.opensuse.org/opensuse-security-...",True,False
82,CVE-2019-5717,2019-01-08T23:29:00.373,In Wireshark 2.6.0 to 2.6.5 and 2.4.0 to 2.4.1...,Modified,5.5,3.0,LOCAL,"[""CWE-20""]","[""cpe:2.3:a:wireshark:wireshark:*:*:*:*:*:*:*:...","[""http://lists.opensuse.org/opensuse-security-...",True,False
83,CVE-2019-5718,2019-01-08T23:29:00.437,In Wireshark 2.6.0 to 2.6.5 and 2.4.0 to 2.4.1...,Modified,5.5,3.0,LOCAL,"[""CWE-125""]","[""cpe:2.3:a:wireshark:wireshark:*:*:*:*:*:*:*:...","[""http://lists.opensuse.org/opensuse-security-...",True,False


### Vulhub 

In [18]:
# Cella 5: Acquisizione dati Vulhub

VULHUB_URL = (
    "https://raw.githubusercontent.com/"
    "vulhub/vulhub/master/environments.toml"
)

vulhub_file = RAW_DIR / "vulhub_environments.toml"

# Download con caching locale
if not vulhub_file.exists():
    response = requests.get(VULHUB_URL, timeout=60)
    response.raise_for_status()
    vulhub_file.write_bytes(response.content)

# Lettura del file TOML
with vulhub_file.open("rb") as f:
    vulhub_data = tomllib.load(f)

vulhub_records = []

for env in vulhub_data["environment"]:
    for cve_id in env.get("cve", []):

        vulhub_records.append({
            "cve_id": cve_id.upper(),
            "vulhub_name": env.get("name"),
            "vulhub_app": env.get("app"),
            "vulhub_path": env.get("path"),
            "vulhub_tags": json.dumps(env.get("tags", [])),
            "vulhub_url": (
                "https://github.com/vulhub/vulhub/tree/master/"
                + env.get("path", "")
            )
        })

df_vulhub = pd.DataFrame(vulhub_records)

df_vulhub.to_csv(
    PROCESSED_DIR / "vulhub.csv", index=False
)

# Corrispondenze con le CVE candidate
vulhub_matches = df_candidates[
    df_candidates["cve_id"].isin(df_vulhub["cve_id"])
]

# Possibili CVE Linux sfuggite al primo filtro
vulhub_recovery = df_unknown[
    df_unknown["cve_id"].isin(df_vulhub["cve_id"])
]

print(f"Ambienti Vulhub: {len(vulhub_data['environment'])}")
print(f"CVE uniche Vulhub: {df_vulhub['cve_id'].nunique()}")
print(f"Corrispondenze con candidate: {len(vulhub_matches)}")
print(f"Possibili CVE da recuperare: {len(vulhub_recovery)}")

display(df_vulhub.head())

Ambienti Vulhub: 334
CVE uniche Vulhub: 274
Corrispondenze con candidate: 52
Possibili CVE da recuperare: 124


,cve_id,vulhub_name,vulhub_app,vulhub_path,vulhub_tags,vulhub_url
0,CVE-2024-39907,1Panel Control Panel PostAuth SQL Injection,1panel,1panel/CVE-2024-39907,"[""SQL Injection""]",https://github.com/vulhub/vulhub/tree/master/1...
1,CVE-2015-5254,Apache ActiveMQ Deserialization,Apache ActiveMQ,activemq/CVE-2015-5254,"[""Deserialization"", ""RCE"", ""Message Queue""]",https://github.com/vulhub/vulhub/tree/master/a...
2,CVE-2016-3088,Apache ActiveMQ Arbitrary File Write,Apache ActiveMQ,activemq/CVE-2016-3088,"[""File Upload"", ""RCE"", ""Message Queue""]",https://github.com/vulhub/vulhub/tree/master/a...
3,CVE-2022-41678,Apache ActiveMQ Jolokia Authenticated Remote C...,Apache ActiveMQ,activemq/CVE-2022-41678,"[""RCE"", ""Message Queue""]",https://github.com/vulhub/vulhub/tree/master/a...
4,CVE-2023-46604,Apache ActiveMQ OpenWire Protocol Deserializat...,Apache ActiveMQ,activemq/CVE-2023-46604,"[""RCE"", ""Deserialization"", ""Message Queue""]",https://github.com/vulhub/vulhub/tree/master/a...


### Metasploit

In [19]:
# Cella 6: Acquisizione dati Metasploit

MSF_URL = (
    "https://raw.githubusercontent.com/rapid7/"
    "metasploit-framework/master/db/modules_metadata_base.json"
)

msf_file = RAW_DIR / "metasploit_metadata.json"

# Download con caching locale
if not msf_file.exists():
    response = requests.get(MSF_URL, timeout=120)
    response.raise_for_status()
    msf_file.write_bytes(response.content)

msf_data = json.loads(
    msf_file.read_text(encoding="utf-8")
)

msf_records = []

for module in msf_data.values():

    references = module.get("references", [])

    cve_ids = set(re.findall(
        r"CVE-\d{4}-\d{4,}",
        json.dumps(references).upper()
    ))

    for cve_id in sorted(cve_ids):

        msf_records.append({
            "cve_id": cve_id,
            "msf_name": module.get("name"),
            "msf_module": module.get("fullname"),
            "msf_type": module.get("type"),
            "msf_platform": module.get("platform"),
            "msf_rank": module.get("rank"),
            "msf_description": module.get("description")
        })

df_msf = pd.DataFrame(msf_records).drop_duplicates()

df_msf.to_csv(
    PROCESSED_DIR / "metasploit.csv", index=False
)

# Moduli classificati come exploit
df_msf_exploits = df_msf[
    df_msf["msf_type"] == "exploit"
].copy()

# Incrocio con le candidate
msf_matches = df_candidates[
    df_candidates["cve_id"].isin(df_msf_exploits["cve_id"])
]

# Possibili CVE da recuperare
msf_recovery = df_unknown[
    df_unknown["cve_id"].isin(df_msf_exploits["cve_id"])
]

print(f"Moduli Metasploit: {len(msf_data)}")
print(f"CVE uniche: {df_msf['cve_id'].nunique()}")
print(f"CVE associate a exploit: {df_msf_exploits['cve_id'].nunique()}")
print(f"Corrispondenze con candidate: {len(msf_matches)}")
print(f"Possibili CVE da recuperare: {len(msf_recovery)}")

display(df_msf.head())

Moduli Metasploit: 7267
CVE uniche: 3248
CVE associate a exploit: 2603
Corrispondenze con candidate: 74
Possibili CVE da recuperare: 821


,cve_id,msf_name,msf_module,msf_type,msf_platform,msf_rank,msf_description
0,CVE-2007-4387,2Wire Cross-Site Request Forgery Password Rese...,auxiliary/admin/2wire/xslt_password_reset,auxiliary,,300,This module will reset the admin password on a...
1,CVE-2014-6041,Android Browser RCE Through Google Play Store XFO,auxiliary/admin/android/google_play_store_uxss...,auxiliary,,300,This module combines two vulnerabilities to ac...
2,CVE-2005-2611,Veritas Backup Exec Windows Remote File Access,auxiliary/admin/backupexec/dump,auxiliary,,300,This module abuses a logic flaw in the Backup ...
3,CVE-2005-0771,Veritas Backup Exec Server Registry Access,auxiliary/admin/backupexec/registry,auxiliary,,300,This modules exploits a remote registry access...
4,CVE-2004-0795,IBM DB2 db2rcmd.exe Command Execution Vulnerab...,auxiliary/admin/db2/db2rcmd,auxiliary,,300,This module exploits a vulnerability in the Re...


## 7. Exploit-DB – Raccolta delle evidenze di exploit

Acquisizione dei metadati degli exploit pubblici e associazione
alle vulnerabilità tramite CVE ID.

In [20]:
# Cella 7: Acquisizione dati Exploit-DB

EDB_URL = (
    "https://gitlab.com/exploit-database/"
    "exploitdb/-/raw/main/files_exploits.csv"
)

edb_file = RAW_DIR / "exploitdb.csv"

# Download con caching locale
if not edb_file.exists():
    response = requests.get(EDB_URL, timeout=120)
    response.raise_for_status()
    edb_file.write_bytes(response.content)

# Lettura del dataset
edb_data = pd.read_csv(edb_file).fillna("")

edb_records = []

for _, row in edb_data.iterrows():

    cve_ids = set(re.findall(
        r"CVE-\d{4}-\d{4,}",
        str(row.get("codes", "")).upper()
    ))

    for cve_id in sorted(cve_ids):

        edb_records.append({
            "cve_id": cve_id,
            "edb_id": row.get("id"),
            "edb_description": row.get("description"),
            "edb_type": row.get("type"),
            "edb_platform": row.get("platform"),
            "edb_verified": row.get("verified"),
            "edb_date": row.get("date_published")
        })

df_edb = pd.DataFrame(edb_records).drop_duplicates()

df_edb.to_csv(
    PROCESSED_DIR / "exploitdb.csv", index=False
)

# Incrocio con le CVE candidate
edb_matches = df_candidates[
    df_candidates["cve_id"].isin(df_edb["cve_id"])
]

# Possibili CVE da recuperare
edb_recovery = df_unknown[
    df_unknown["cve_id"].isin(df_edb["cve_id"])
]

print(f"Exploit indicizzati: {len(edb_data)}")
print(f"CVE uniche: {df_edb['cve_id'].nunique()}")
print(f"Corrispondenze con candidate: {len(edb_matches)}")
print(f"Possibili CVE da recuperare: {len(edb_recovery)}")

display(df_edb.head())

Exploit indicizzati: 47182
CVE uniche: 25095
Corrispondenze con candidate: 140
Possibili CVE da recuperare: 1956


,cve_id,edb_id,edb_description,edb_type,edb_platform,edb_verified,edb_date
0,CVE-2009-3699,16929,AIX Calendar Manager Service Daemon (rpc.cmsd)...,dos,aix,1,2010-11-11
1,CVE-1999-1015,19046,AppleShare IP Mail Server 5.0.3 - Buffer Overflow,dos,aix,1,1999-10-15
2,CVE-2003-0087,22249,IBM AIX 4.3.3/5.1/5.2 - 'libIM' Buffer Overflow,dos,aix,1,2003-02-12
3,CVE-2009-4265,16657,PointDev IDEAL Migration - Buffer Overflow (Me...,dos,aix,1,2010-09-25
4,CVE-2014-9349,35342,RobotStats 1.0 - HTML Injection,dos,aix,0,2014-11-24


## 8. GitHub – Raccolta dei riferimenti pubblici

Estrazione e classificazione dei riferimenti GitHub
associati alle CVE attraverso i dati NVD.

In [21]:
# Cella 8: Acquisizione riferimenti GitHub

from urllib.parse import urlparse

github_records = []

df_search = pd.concat([
    df_candidates[["cve_id", "references"]],
    df_unknown[["cve_id", "references"]]
], ignore_index=True)

for row in df_search.itertuples(index=False):

    references = json.loads(row.references)

    for url in references:

        parsed = urlparse(url)

        if parsed.hostname not in ["github.com", "www.github.com"]:
            continue

        parts = parsed.path.strip("/").split("/")

        if len(parts) < 2:
            continue

        repository = "/".join(parts[:2])
        reference_type = "other"

        if "advisories" in parts:
            reference_type = "advisory"

        elif "commit" in parts:
            reference_type = "commit"

        elif "issues" in parts:
            reference_type = "issue"

        elif "pull" in parts:
            reference_type = "pull_request"

        elif len(parts) == 2:
            reference_type = "repository"

        elif "blob" in parts:
            reference_type = "file"

        github_records.append({
            "cve_id": row.cve_id,
            "github_repository": repository,
            "github_url": url,
            "github_type": reference_type
        })

df_github = pd.DataFrame(
    github_records,
    columns=[
        "cve_id", "github_repository",
        "github_url", "github_type"
    ]
).drop_duplicates()

df_github.to_csv(
    PROCESSED_DIR / "github_references.csv", index=False
)

github_matches = df_candidates[
    df_candidates["cve_id"].isin(df_github["cve_id"])
]

github_recovery = df_unknown[
    df_unknown["cve_id"].isin(df_github["cve_id"])
]

print(f"Riferimenti GitHub: {len(df_github)}")
print(f"CVE uniche: {df_github['cve_id'].nunique()}")
print(f"Corrispondenze con candidate: {len(github_matches)}")
print(f"Corrispondenze con piattaforma incerta: {len(github_recovery)}")

print("\nDistribuzione dei riferimenti:")
print(df_github["github_type"].value_counts())

display(df_github.head())

Riferimenti GitHub: 147087
CVE uniche: 79316
Corrispondenze con candidate: 3756
Corrispondenze con piattaforma incerta: 75560

Distribuzione dei riferimenti:
github_type
commit          31843
advisory        30729
file            23621
other           20850
issue           18311
pull_request    11589
repository      10144
Name: count, dtype: int64


,cve_id,github_repository,github_url,github_type
0,CVE-2019-3500,aria2/aria2,https://github.com/aria2/aria2/issues/1329,issue
2,CVE-2019-0542,xtermjs/xterm.js,https://github.com/xtermjs/xterm.js/releases,other
4,CVE-2019-5882,irssi/irssi,https://github.com/irssi/irssi/pull/948,pull_request
6,CVE-2019-5892,FRRouting/frr,https://github.com/FRRouting/frr/commit/943d59...,commit
7,CVE-2019-5892,FRRouting/frr,https://github.com/FRRouting/frr/releases/tag/...,other


## 9. Falco Rules – Raccolta dei riferimenti di detection

Acquisizione delle regole ufficiali Falco e identificazione
delle associazioni esplicite con identificativi CVE.

In [22]:
# Cella 9: Acquisizione regole Falco

FALCO_BASE_URL = (
    "https://raw.githubusercontent.com/"
    "falcosecurity/rules/main/rules/"
)

FALCO_FILES = {
    "stable": "falco_rules.yaml",
    "incubating": "falco-incubating_rules.yaml",
    "sandbox": "falco-sandbox_rules.yaml"
}

falco_records = []
total_rules = 0

for maturity, filename in FALCO_FILES.items():

    file_path = RAW_DIR / filename

    # Download con caching locale
    if not file_path.exists():
        response = requests.get(
            FALCO_BASE_URL + filename, timeout=60
        )
        response.raise_for_status()
        file_path.write_bytes(response.content)

    # Lettura del file YAML
    rules = yaml.safe_load(
        file_path.read_text(encoding="utf-8")
    )

    for rule in rules:

        if not isinstance(rule, dict) or "rule" not in rule:
            continue

        total_rules += 1

        cve_ids = set(re.findall(
            r"CVE-\d{4}-\d{4,}",
            json.dumps(rule).upper()
        ))

        for cve_id in sorted(cve_ids):

            falco_records.append({
                "cve_id": cve_id,
                "falco_rule": rule.get("rule"),
                "falco_description": rule.get("desc"),
                "falco_priority": rule.get("priority"),
                "falco_maturity": maturity,
                "falco_source": rule.get("source", "syscall"),
                "falco_url": (
                    "https://github.com/falcosecurity/"
                    "rules/blob/main/rules/" + filename
                )
            })

df_falco = pd.DataFrame(
    falco_records,
    columns=[
        "cve_id", "falco_rule", "falco_description",
        "falco_priority", "falco_maturity",
        "falco_source", "falco_url"
    ]
).drop_duplicates()

df_falco.to_csv(
    PROCESSED_DIR / "falco_rules.csv", index=False
)

falco_matches = df_candidates[
    df_candidates["cve_id"].isin(df_falco["cve_id"])
]

falco_recovery = df_unknown[
    df_unknown["cve_id"].isin(df_falco["cve_id"])
]

print(f"Regole Falco analizzate: {total_rules}")
print(f"Associazioni CVE-regola: {len(df_falco)}")
print(f"CVE uniche: {df_falco['cve_id'].nunique()}")
print(f"Corrispondenze con candidate: {len(falco_matches)}")
print(f"Possibili CVE da recuperare: {len(falco_recovery)}")

display(df_falco.head())

Regole Falco analizzate: 96
Associazioni CVE-regola: 6
CVE uniche: 6
Corrispondenze con candidate: 4
Possibili CVE da recuperare: 1


,cve_id,falco_rule,falco_description,falco_priority,falco_maturity,falco_source,falco_url
0,CVE-2020-14386,Packet socket created in container,Detect new packet socket at the device driver ...,NOTICE,stable,syscall,https://github.com/falcosecurity/rules/blob/ma...
1,CVE-2024-3094,Backdoored library loaded into SSHD (CVE-2024-...,This rule detects possible CVE-2024-3094 explo...,WARNING,incubating,syscall,https://github.com/falcosecurity/rules/blob/ma...
2,CVE-2021-3156,Sudo Potential Privilege Escalation,"Affecting sudo (<= 1.9.5p2), there's a privile...",CRITICAL,sandbox,syscall,https://github.com/falcosecurity/rules/blob/ma...
3,CVE-2021-4034,Polkit Local Privilege Escalation Vulnerabilit...,This rule detects attempts to exploit a privil...,CRITICAL,sandbox,syscall,https://github.com/falcosecurity/rules/blob/ma...
4,CVE-2021-44228,Java Process Class File Download,Detecting a Java process downloading a class f...,CRITICAL,sandbox,syscall,https://github.com/falcosecurity/rules/blob/ma...
